# Configurable NLP Experimentation Workbench
### Hardware-Aware NLP Research, Rapid Prototyping & Gemastik Data Mining

An end-to-end, modular, hardware-aware NLP experimentation framework designed for:
1. **Gemastik Data Mining Preparation**: Strict competition pipeline, unlabelled test evaluation, ID alignment, and submission verification.
2. **Reproducible NLP Research**: Fold-local preprocessing, leak-free cross-validation, held-out test isolation, and explainable decision support.
3. **Hardware Efficiency**: Tailored for NVIDIA GeForce RTX 3050 (4GB/6GB VRAM) and CPU-only environments with dynamic batch sizing and mixed precision.
4. **Architecture Diversity**: 14 task categories × 3 executable approaches = 42 real model implementations integrated via the Architecture Registry.


## 00. Environment Setup & Package Verification
Verify that `nlp_workbench` and required machine learning dependencies are installed.


In [ ]:
import sys
from pathlib import Path

# Verify and import nlp_workbench package
try:
    import nlp_workbench
    print(f"nlp_workbench version: {nlp_workbench.__version__}")
except ImportError:
    # If running directly from git root without pip install -e .
    sys.path.insert(0, str(Path("src").resolve()))
    import nlp_workbench
    print(f"nlp_workbench version: {nlp_workbench.__version__} (loaded from src/)")


## 01. Configuration — Experiment Control Center
Configure the experimental setup in one centralized location.

### Supported Tasks:
`classification`, `regression`, `multilabel`, `ner`, `qa`, `summarization`, `translation`, `generation`, `similarity`, `retrieval`, `topic_modeling`, `pair_classification`, `zeroshot`.


In [ ]:
from nlp_workbench import WorkbenchConfig, get_default_config

# Initialize configuration
# To use your own dataset, set dataset_path="path/to/dataset.csv"
config = get_default_config(task="classification", language="indonesian")

config.dataset_path = "synthetic"  # Change to your dataset file path or "synthetic"
config.text_column = "text"
config.target_column = "target"
config.device = "auto"             # "auto", "cuda", or "cpu"
config.split_strategy = "stratified"
config.n_splits = 5
config.model_ids = None            # None for auto-selection, or list like ["classical_nb", "classical_logistic"]
config.gemastik_mode = False       # Set to True when participating in Gemastik Data Mining

# Display active configuration
print("Active Workbench Configuration:")
print(f"  Task:            {config.task}")
print(f"  Language:        {config.language}")
print(f"  Dataset Path:    {config.data_path}")
print(f"  Text Column(s):  {config.text_columns}")
print(f"  Target Column:   {config.target_column}")
print(f"  Target Device:   {config.device}")
print(f"  Selected Models: {config.model_ids or "Auto-select top compatible"}")
print(f"  CV Folds:        {config.n_splits}")
print(f"  Gemastik Mode:   {config.gemastik_mode}")


## 02. Hardware Profiling & VRAM Diagnostics
Inspects host hardware (CPU, CUDA, GPU model, VRAM limits) and calculates safe batch sizes and gradient accumulation steps for RTX 3050 (4GB) or CPU systems.


In [ ]:
from nlp_workbench import HardwareManager

# Inspect host hardware
hw = HardwareManager.inspect(config.device)
print(hw.summary())

# Hardware recommendations
rec = HardwareManager(config.device).recommend_batch_size()
print(f"\nHardware Manager Recommended Settings:")
print(f"  Batch Size:             {rec['batch_size']}")
print(f"  Grad Accumulation Steps:{rec['grad_accum_steps']}")


## 03. Dataset Ingestion & Schema Validation
Loads tabular data (CSV, TSV, Parquet, JSONL) or synthetic benchmark data with strict schema validation.
Never silently fabricates data if a real dataset path is specified.


In [ ]:
from nlp_workbench import DataLoader, validate_dataset_schema

if config.data_path == "synthetic":
    print("Generating synthetic benchmark dataset for task:", config.task)
    df = DataLoader.generate_synthetic_dataset(
        task=config.task,
        n_samples=150,
        language=config.language,
        random_state=config.random_state,
    )
    # Align column names to config
    if config.task == "classification":
        df = df.rename(columns={"content": config.text_column, "category": config.target_column})
else:
    df = DataLoader.load_data(config.data_path)

print(f"Dataset Loaded Successfully: {df.shape[0]} rows, {df.shape[1]} columns")

# Validate schema
is_valid, errors = validate_dataset_schema(df, config)
if not is_valid:
    print("Schema Warnings/Errors Detected:")
    for err in errors:
        print(f"  - {err}")
else:
    print("Schema Validation Passed: Required columns exist and contain valid labels.")

df.head(5)


## 04 & 05. Exploratory Data Analysis & 5-Point Explainable Decision Support
Audits dataset quality and provides actionable 5-point decision support:
1. **What is the issue**
2. **Evidence observed**
3. **Why it matters**
4. **Recommended action**
5. **Potential drawbacks & controlling configuration variable**


In [ ]:
from nlp_workbench import DataQualityAuditor

auditor = DataQualityAuditor(
    df=df,
    text_columns=config.text_columns,
    target_column=config.target_column,
    id_column=config.id_column,
    task=config.task,
)

summary_stats, findings = auditor.audit()

print("--- Column-by-Column Profiling ---")
print(summary_stats.to_string())

print("\n--- Actionable Decision Board ---")
decision_df = auditor.decision_board()
if not decision_df.empty:
    display_cols = ["Severity", "Finding", "Why It Matters", "Recommended Action", "Action Key"]
    print(decision_df[display_cols].to_string(index=False))
else:
    print("No critical data quality issues identified.")


## 06 & 07. Preprocessing & Non-Destructive Cleaning
Recommends preprocessing configurations based on task, model family, and language.
Applies negation-preserving Indonesian stopword filtering and HTML/URL normalization.


In [ ]:
from nlp_workbench import recommend_preprocessing, TextCleaner

# Obtain automated recommendation
rec = recommend_preprocessing(task=config.task, model_family="classical", language=config.language)
print("Automated Preprocessing Recommendation:")
print(f"  Lowercase:              {rec.lowercase}")
print(f"  Remove HTML / URLs:     {rec.remove_html} / {rec.handle_urls}")
print(f"  Remove Stopwords:       {rec.remove_stopwords}")
print(f"  Preserve Negations:     {rec.preserve_negations}")
print(f"  N-Gram Range:           {rec.ngram_range}")
print(f"  Rationale:              {rec.rationale}")

# Instantiate TextCleaner
cleaner = TextCleaner(
    remove_html=config.accepted_decisions.get("enable_html_cleaning", True),
    handle_urls=config.accepted_decisions.get("handle_urls", True),
    lowercase=True,
    language=config.language,
)

# Test cleaning on sample text
sample_raw = "<p>Pelayanan rumah sakit ini TIDAK ramah sama sekali! Kunjungi https://rs-info.id</p>"
sample_cleaned = cleaner.clean_text(sample_raw)
print(f"\nSample Raw:     {sample_raw}")
print(f"Sample Cleaned: {sample_cleaned}")


## 08. Dataset Splitting & Strict Leakage Auditing
Splits the dataset into untouched Train, Validation, and Held-Out Test sets.
Performs rigorous data leakage auditing for:
- Duplicate text across splits
- Overlapping sample IDs
- Contaminated author / document groups


In [ ]:
from nlp_workbench import DatasetSplitter, LeakageAuditor

splitter = DatasetSplitter(
    strategy=config.split_strategy,
    n_splits=config.n_splits,
    test_size=config.experimental_preferences.get("test_size", 0.2),
    random_state=config.random_state,
)

train_df, val_df, test_df = splitter.split(
    df=df,
    target_col=config.target_column,
    task=config.task,
)

print(f"Dataset Partitions Created:")
print(f"  Train Set:        {len(train_df)} samples ({len(train_df)/len(df)*100:.1f}%)")
print(f"  Validation Set:   {len(val_df)} samples ({len(val_df)/len(df)*100:.1f}%)")
print(f"  Held-Out Test Set:{len(test_df)} samples ({len(test_df)/len(df)*100:.1f}%)")

# Audit data leakage between train and test sets
auditor_leakage = LeakageAuditor()
leak_report = auditor_leakage.audit_splits(
    train_df=train_df,
    test_df=test_df,
    text_columns=config.text_columns,
    id_column=config.id_column,
)

print(f"\nData Leakage Audit Results:")
print(f"  Duplicate Texts Found: {leak_report['duplicate_text_count']} ({leak_report['duplicate_text_pct']}%)")
print(f"  Overlapping IDs Found: {leak_report['id_overlap_count']}")
print(f"  Leakage Detected:      {leak_report['leakage_detected']}")


## 09 & 10. Architecture Registry (14 Categories × 3 Models = 42 Approaches)
Every selectable model is backed by a callable execution adapter with lazy loading.
Inspect the models available for the active task.


In [ ]:
from nlp_workbench import registry

print(f"Total Models in Architecture Registry: {len(registry.models)}")

# Filter models suitable for current task and language
task_models = registry.list_models(task=config.task, language=config.language)
print(f"Models Compatible with '{config.task}' in '{config.language}': {len(task_models)}\n")

df_models = registry.to_dataframe(task=config.task, language=config.language)
print(df_models[["model_id", "category", "approach_name", "framework", "vram_tier"]].to_string(index=False))


## 11. Model Training & Fold-Local Cross-Validation
Executes cross-validation on the training set.
**Methodological Integrity**: Preprocessing (TF-IDF vectorizer, scalers) is fit strictly within each fold slice to guarantee zero feature leakage.


In [ ]:
from nlp_workbench import run_cross_validation, registry

# Dynamically resolve models from config or architecture registry
if config.model_ids and len(config.model_ids) > 0:
    candidate_models = list(config.model_ids)
else:
    compatible = registry.get_compatible_models(task=config.task, language=config.language)
    if not compatible:
        compatible = registry.list_models(task=config.task)
    candidate_models = [m.model_id for m in compatible[:3]]

print(f"Candidate Models for Task '{config.task}': {candidate_models}")
print()

leaderboard = []
model_adapters = {}
fold_adapters_dict = {}

print("Starting Model Training & Validation:")
for model_id in candidate_models:
    rec = registry.get_model(model_id)
    adapter = registry.create_adapter(
        model_id,
        text_columns=config.text_columns,
        target_column=config.target_column,
        device=config.device,
        **config.task_params,
    )

    if not rec.is_inference_only and "training" in rec.supported_operations:
        cv_res = run_cross_validation(
            adapter=adapter,
            train_df=train_df,
            task=config.task,
            target_column=config.target_column,
            text_columns=config.text_columns,
            n_splits=config.n_splits,
            random_state=config.random_state,
            group_column=config.group_column,
        )
        fold_adapters_dict[model_id] = cv_res.get("fold_adapters", [])
        leaderboard.append(cv_res)
        adapter.fit(train_df, val_df=val_df, target_column=config.target_column)
        model_adapters[model_id] = adapter
        print(f"  [{model_id}] Mean {cv_res['primary_metric']}: {cv_res['mean_cv_score']:.4f} (+/- {cv_res['std_cv_score']:.4f}) | Duration: {cv_res['cv_duration_s']}s")
    else:
        # Pretrained / zero-shot / inference-only models
        adapter.fit(train_df, val_df=val_df)
        val_metrics = adapter.evaluate(val_df, target_column=config.target_column)
        metric_name = list(val_metrics.keys())[0] if val_metrics else "Score"
        score = float(val_metrics.get(metric_name, 0.0))
        entry = {
            "model_id": model_id,
            "mean_cv_score": score,
            "std_cv_score": 0.0,
            "primary_metric": metric_name,
            "cv_duration_s": 0.0,
            "val_metrics": val_metrics,
        }
        leaderboard.append(entry)
        model_adapters[model_id] = adapter
        print(f"  [{model_id}] Val {metric_name}: {score:.4f} (Inference-only)")


## 12. Unified Model Comparison & Leaderboard
Aggregates cross-validation metrics across all evaluated architectures into a clean comparison table.


In [ ]:
import pandas as pd
from nlp_workbench.evaluation import is_higher_better

board_rows = []
for entry in leaderboard:
    board_rows.append({
        "Model ID": entry["model_id"],
        "CV Mean Score": round(float(entry["mean_cv_score"]), 4),
        "CV Score Std": round(float(entry["std_cv_score"]), 4),
        "Primary Metric": entry["primary_metric"],
        "CV Duration (s)": entry["cv_duration_s"],
    })

first_metric = board_rows[0]["Primary Metric"] if board_rows else "Score"
hib = is_higher_better(first_metric)
leaderboard_df = pd.DataFrame(board_rows).sort_values(by="CV Mean Score", ascending=(not hib)).reset_index(drop=True)
print("=== Validation / Cross-Validation Leaderboard ===")
print(leaderboard_df.to_string(index=False))


## 13. Final Model Selection & Held-Out Test Evaluation
**Scientific Rigor**:
1. The winning architecture is selected **strictly** using validation / CV scores.
2. The held-out test set is evaluated **exactly once** on the selected model to estimate true generalization error.


In [ ]:
from nlp_workbench import select_best_model

# Select winning model strictly on validation / CV scores (never on test)
best_model_id, best_cv_score = select_best_model(leaderboard)
print(f"Selected Winning Architecture: {best_model_id} (Score: {best_cv_score:.4f})")

final_adapter = model_adapters[best_model_id]

# Single evaluation on untouched held-out test partition
test_metrics = final_adapter.evaluate(test_df, target_column=config.target_column)
print()
print("Held-Out Test Evaluation (Evaluated Once):")
for metric_name, val in test_metrics.items():
    if isinstance(val, (int, float)):
        print(f"  {metric_name}: {val:.4f}")
    elif isinstance(val, list) and len(val) < 10:
        print(f"  {metric_name}: {val}")


## 14. Error Analysis & Diagnostics
Examines misclassified samples and investigates prediction confidence.


In [ ]:
import numpy as np

test_preds = final_adapter.predict(test_df)
test_df_eval = test_df.copy()
test_df_eval["prediction"] = test_preds

if config.task == "classification" and config.target_column in test_df.columns:
    test_df_eval["is_correct"] = (test_df_eval[config.target_column].astype(str) == test_df_eval["prediction"].astype(str))
    acc = test_df_eval["is_correct"].mean()
    print(f"Held-Out Test Accuracy: {acc * 100:.2f}%")
    misclassified = test_df_eval[~test_df_eval["is_correct"]]
    print(f"Misclassified Samples: {len(misclassified)} of {len(test_df_eval)}")
    if not misclassified.empty:
        sample_errors = misclassified[[config.text_column, config.target_column, "prediction"]].head(5)
        print()
        print("Sample Misclassifications:")
        print(sample_errors.to_string(index=False))
elif config.task == "regression" and config.target_column in test_df.columns:
    residuals = np.abs(test_df[config.target_column].values - np.array(test_preds))
    print(f"Mean Absolute Error on Test: {np.mean(residuals):.4f}")
    print(f"Max Residual on Test: {np.max(residuals):.4f}")
else:
    print(f"Error diagnostics completed for task '{config.task}'.")


## 15. Complete Model Packaging & Export
Exports the winning adapter, complete with vectorizers, encoders, metadata, and preprocessing pipeline, into a self-contained artifact.


In [ ]:
from nlp_workbench import ModelExporter

metadata = {
    "task": config.task,
    "language": config.language,
    "best_model_id": best_model_id,
    "cv_score": best_cv_score,
    "test_metrics": test_metrics,
    "classes": list(final_adapter.classes_) if hasattr(final_adapter, "classes_") else [],
}

bundle_path = ModelExporter.export(
    adapter=final_adapter,
    output_dir=config.output_dir,
    task=config.task,
    metadata=metadata,
    cleaner_config=cleaner.__dict__,
)

print(f"Model Artifact Exported Successfully:")
print(f"  Bundle File/Dir: {bundle_path.resolve()}")


## 16. Standalone Production Inference (`predict_new`)
Provides a single-line production inference function that accepts raw strings, lists of texts, or DataFrames and returns predictions and confidence scores.


In [ ]:
from nlp_workbench import predict_new

# Load exported bundle for standalone inference
loaded_bundle = ModelExporter.load(bundle_path)

sample_queries = [
    "Pemerintah mengumumkan kebijakan fiskal baru untuk mengatasi inflasi perbankan",
    "Tim dokter berhasil menyelesaikan uji klinis vaksin terbaru dengan aman",
    "Startup teknologi meluncurkan komputasi cloud berbasis kecerdasan buatan",
]

result = predict_new(
    texts=sample_queries,
    adapter=loaded_bundle,
)

print("--- Standalone Production Inference Results ---")
print(result.df[["input_text", "prediction", "confidence"]].to_string(index=False))


## 17. Gemastik Data Mining Competition Mode
When `gemastik_mode=True`, ingests the unlabelled competition test set, performs inference, strictly verifies row preservation and ID alignment, and outputs `submission.csv`.


In [ ]:
from nlp_workbench import GemastikRunner

if config.gemastik_mode:
    print("=== Gemastik Data Mining Competition Runner ===")
    runner = GemastikRunner(
        id_column=config.id_column,
        target_column=config.target_column,
        task=config.task,
    )

    # Ingest competition test set
    if config.competition_test_path and config.competition_test_path != "synthetic":
        test_source = config.competition_test_path
    else:
        # Create unlabelled competition test set from test_df
        test_source = test_df[[config.id_column, config.text_column]].copy()

    comp_test_df = runner.prepare_test_dataset(test_source)
    print(f"Competition Test Set Ingested: {len(comp_test_df)} rows")

    # Generate competition predictions with fold-ensemble if available
    fold_models = fold_adapters_dict.get(best_model_id, [])
    comp_preds = runner.generate_predictions(
        adapter=final_adapter,
        test_df=comp_test_df,
        text_column=config.text_column,
        ensemble_adapters=fold_models if len(fold_models) > 1 else None,
    )

    # Create and strictly validate submission file
    submission_summary = runner.create_submission(
        test_df=comp_test_df,
        predictions=comp_preds,
        output_path=config.submission_path,
    )

    print(f"Submission Created & Validated Successfully:")
    print(f"  Output Path:   {submission_summary['submission_path']}")
    print(f"  Rows Count:    {submission_summary['row_count']}")
    print(f"  Columns:       {submission_summary['columns']}")
    print(f"  Head Preview:  {submission_summary['sample_head']}")
else:
    print("Gemastik competition mode is disabled (config.gemastik_mode = False).")
    print("Set config.gemastik_mode = True to generate a competition submission file.")


## 18. Experiment Summary & Reproducibility Audit
Generates a comprehensive audit record of the experiment including git commit, dependencies, hyperparameters, and environment settings.


In [ ]:
import json
import datetime
import subprocess

def get_git_commit():
    try:
        return subprocess.check_output(["git", "rev-parse", "HEAD"]).decode("ascii").strip()
    except Exception:
        return "N/A"

audit_record = {
    "timestamp": datetime.datetime.now().isoformat(),
    "git_commit": get_git_commit(),
    "config": config.to_dict(),
    "hardware": {
        "device": hw.device,
        "cuda_available": hw.cuda_available,
        "gpu_name": hw.gpu_name,
    },
    "best_model_id": best_model_id,
    "cv_score": best_cv_score,
    "test_metrics": {k: float(v) for k, v in test_metrics.items() if isinstance(v, (int, float))},
    "submission_path": str(Path(config.submission_path).resolve()),
}

audit_file = Path(config.output_dir) / "experiment_audit.json"
with open(audit_file, "w") as f:
    json.dump(audit_record, f, indent=2)

print(f"Reproducibility Audit Saved to: {audit_file.resolve()}")
print(f"Experiment completed successfully!")
